# BONSAI parser walkthrough

This notebook shows how to parse BONSAI's Balanced hybrid supply-use table (HSUT) and native b2 hybrid input-output table (HIOT). Monetary SUT exports are not supported.
The checked source directory is `20260615`, with reference year **2016**. Its name does not identify an immutable snapshot: individual files can be updated within it.


## What this notebook covers

- which local files the parser expects;
- how to read a Balanced HSUT or native b2 HIOT;
- how to inspect regional classifications and native units;
- where to find import diagnostics and further analysis methods.

Jump to [Balanced HSUT](#Parse-the-HSUT) or [native b2 HIOT](#Parse-the-native-b2-HIOT).


## Relevant source pages

- [BONSAI project](https://bonsai.uno/)
- [BONSAI repository](https://gitlab.com/bonsamurais/bonsai)

Obtain the bundle from the project's shared data storage, under `_b Next version/Balanced` (HSUT) or `_b Next version/HIOT` (HIOT).
MARIO does not download BONSAI automatically.

## Main entry point

Use `mario.parse_bonsai(...)` with the original ZIP or its extracted directory. The default `table="SUT"` reads the Balanced HSUT; `table="IOT"` reads the native b2 HIOT demonstrated below.


## Key arguments and local layout

- `path`: a ZIP or extracted directory for the requested table;
- `table`: `"SUT"` (default) for the Balanced HSUT or `"IOT"` for the native b2 HIOT;
- `extensions`: `True` (default) includes the recalculated elementary flows; `False` reads only the economic tables.

For the Balanced HSUT, the parser reads these files:

```text
20260615/
├── classification.csv
├── hybrid_supply_w_characteristic_unit.csv
├── hybrid_use_w_characteristic_unit.csv
└── all_elementary_flows_AB_and_BB.csv
```

The last file is needed only when `extensions=True`. Other files in the bundle can be left in place.
See [mario.parse_bonsai](../../../api_document/mario.parse_bonsai.rst) for all arguments.

In [1]:
import mario

mario.set_log_verbosity("warning")

## Parse the HSUT

Pass the ZIP directly; no manual extraction is needed. You can also pass the extracted `20260615` directory.

### Recommended solver for full-database calculations

Install the optional **UMFPACK** backend for calculations on the full database:

```sh
conda install -c conda-forge scikit-umfpack
```

For pip, install **SuiteSparse** with your system package manager first, then
run `python -m pip install "mariopy[umfpack]"`. The Python dependency is
`scikit-umfpack>=0.4.2`; pip does not install SuiteSparse.

A successful import selects `solve` and `direct`, preferring UMFPACK and
using SciPy if the optional backend is unavailable. The import log reports
which solver was selected. These settings apply to the current Python
session, including subsequent calculations on other databases. Inspect them
with `mario.download_settings()["compute"]` or change them using the compute
setting functions. Keep `calc_all=False` to calculate only requested matrices.
See [compute presets and installation](../../../reference/api_settings_methods.rst#compute-presets-for-large-databases).

BONSAI retains `singular_policy="partial"`: affected coefficients are reported as NaN with a warning. This applies with either backend. See the [API notes](../../../api_document/mario.parse_bonsai.rst).


In [2]:
db = mario.parse_bonsai(
    path="/path/to/Balanced/20260615.zip",
)

db

/path/to/MARIO/mario/parsers/entrypoints.py:217: UserWarning: BONSAI: 362 elementary-flow rows have no classified activity/final demand and are omitted; examples: [('AT', 'A_MA_RECY'), ('BE', 'A_MA_RECY'), ('BE', 'A_SEWAGE'), ('BG', 'A_MA_LAND'), ('BG', 'A_MA_OTHE')]. Full rows are in the bonsai import report.
  matrices, indexes, units, layout = parse_bonsai_hsut(


name = BONSAI Balanced HSUT
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = from 365 to 1272 per region
Commodity = from 365 to 1335 per region
Factor of production = 1
Satellite account = 127
Consumption category = from 0 to 6 per region
Region = 50

## Inspect the parsed database

The result is a MARIO SUT database. Check its regions and the native commodity units before starting an analysis.

In [3]:
db.get_index("Region")

['AT',
 'AU',
 'BE',
 'BG',
 'BR',
 'CA',
 'CH',
 'CN',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'ES',
 'FI',
 'FR',
 'GB',
 'GR',
 'Global',
 'HR',
 'HU',
 'ID',
 'IE',
 'IN',
 'IT',
 'JP',
 'KR',
 'LT',
 'LU',
 'LV',
 'MT',
 'MX',
 'NL',
 'NO',
 'PL',
 'PT',
 'RO',
 'RU',
 'SE',
 'SI',
 'SK',
 'TR',
 'TW',
 'US',
 'WA',
 'WE',
 'WF',
 'WL',
 'WM',
 'ZA']

In [ ]:
db.units["Commodity"].head()


## Parse the native b2 HIOT

Use `table="IOT"` with the extracted `HIOT/20260615` directory or a ZIP containing one release. The parser imports the final Stone/iLUC/capital model directly.

```text
20260615/
├── extended_size_hiot.b2.pkl
├── final_demand.b2.csv
└── total_elementary_flow_w_iLUC.csv
```

The elementary-flow file is needed only when `extensions=True`. Only use a trusted BONSAI pickle export.

The reference year is inferred from elementary-flow `time` values; pass `year=2016` if those values are unavailable. Original product codes, native units and region-specific Sector lists are retained. Inspect a single region with `get_index("Sector", region=...)`; the activity-to-reference-product mapping is in `db_iot.info["bonsai"]["reference_products"]`.


In [2]:
from pathlib import Path
import mario

hiot_path = Path("/path/to/HIOT/20260615")
db_iot = mario.parse_bonsai(path=hiot_path, table="IOT")

sectors_it = db_iot.get_index("Sector", region="IT")
print(f"Reference year: {db_iot.meta.year}")
print(f"Native production nodes: {db_iot.X.shape[0]:,}")
print(f"Sectors in Italy: {len(sectors_it)}")
print("First Italian sectors:", sectors_it[:8])


Reference year: 2016
Native production nodes: 53,789
Sectors in Italy: 1230
First Italian sectors: ['C_ACCOM|C_RESTAU', 'C_ACT_HOUS', 'C_ADVERT', 'C_ALUM_01_RES', 'C_ALUW', 'C_ALU_sort', 'C_AL_INCI', 'C_AL_LAND']


## Notes

- Activity, commodity and Sector classifications can differ between regions. See [regional classifications](../../../get_started/nomenclature.rst) and [Basic inspections](../../../user_guide/inspection/basic_inspections.ipynb) for the index API.
- Value-added accounts are not available in these inputs; MARIO uses zero placeholders.
- Satellite labels use `name | compartment`; IDs disambiguate homonyms. Units and native identities remain in `db.units["Satellite account"]` and `db.info["bonsai"]["satellites"]` (or the corresponding `db_iot` attributes).
- Import diagnostics are available in `db.info["bonsai"]`. Unmatched elementary-flow rows generate a warning by default; use `unmatched_extensions="raise"` to reject them. For the HIOT, `unmatched_final_demand` controls the analogous final-demand policy.
- The HIOT preserves the native net-supply model through `X = diag(T)` and `Z = diag(X) - T`, including substitution credits. See the [parser API](../../../api_document/mario.parse_bonsai.rst) for representation and coverage details.
- Keep the default `calc_all=False` when loading a full database. Request only the matrices needed for your analysis; full inverse matrices can require substantial memory.
- For subsequent calculations, see the [matrices API](../../../reference/api_resolvable_matrices.rst) and [GHG characterization](../../../api_document/mario.Database.calc_ghg.rst). Per-unit satellite multipliers are exposed as `fc` for HSUT commodities and `f` for HIOT sectors, in the native product units.
